In [ ]:
import pandas as pd
import numpy as np

In [ ]:
plaintexts = pd.read_csv("Processed Data/cleaned_data.csv")
formats = pd.read_csv("Processed Data/formats.csv")

ATTRIBUTES = plaintexts.columns.tolist()

data = plaintexts.join(formats, lsuffix="_x", rsuffix="_f")

In [ ]:
def leakage_statistics(data, attributes):
    """
    Calculate attribute leakage statistics for the given attributes, including min and shannon entopy.
    Args:
        data (pd.DataFrame): The combined dataframe containing both plaintext and format columns.
        attributes (list of str): The attributes for which to calculate leakage statistics.
    Returns:
        pd.DataFrame: A dataframe containing the leakage statistics for the specified attribute.
    """
    x_cols = [col + "_x" for col in attributes]
    f_cols = [col + "_f" for col in attributes]
    cols = x_cols + f_cols
    d = data[cols].replace("", pd.NA).dropna() 
    x = d[x_cols].apply(tuple, axis=1) if len(x_cols) > 1 else d[x_cols[0]] # Plaintexts
    f = d[f_cols].apply(tuple, axis=1) if len(f_cols) > 1 else d[f_cols[0]] # Formats

    n = len(d)

    # Format class probabilities
    f_counts = f.value_counts()
    p_f = f_counts / n

    # Plaintext min-entropy
    h_inf = -np.log2(x.value_counts(normalize=True).max())
    joint_counts = pd.DataFrame({"x": x, "f": f}).groupby(["f", "x"]).size()
    vuln = joint_counts.groupby(level="f").max().sum() / n
    h_inf_cond = -np.log2(vuln)
    
    return {
        "Total": n,
        "Distinct": len(f_counts),
        "Unique": (f_counts == 1).sum(),
        "Pct": (f_counts == 1).sum() / n * 100,
        "Shannon": -(p_f * np.log2(p_f)).sum(),
        "Min": h_inf - h_inf_cond
    }

In [ ]:
rows = {}
for col in ATTRIBUTES:
    rows[col] = leakage_statistics(data, [col])
rows["NAME + CITY"] = leakage_statistics(data, ["NAME", "CITY"])
rows["NAME + CITY + ZIP"] = leakage_statistics(data, ["NAME", "CITY", "ZIP_CODE"])
rows["NAME + ZIP"] = leakage_statistics(data, ["NAME", "ZIP_CODE"])

statistics = pd.DataFrame(rows).T
statistics = statistics.astype({"Total": int, "Distinct": int, "Unique": int})
print(statistics.round(2))